# The Method of Least Squares, step by step

**What problem does it solve?**
We have $N$ measured points $(x_i, y_i)$ (sensor readings, voltage against temperature, anything) and we believe that,
apart from noise, they follow a straight line:

$$\hat y = w\,x + b$$

$w$ is the slope and $b$ is the intercept. Least squares is a recipe for picking **the** $w$ and $b$ that describe the data best.

**What does "best" mean?** For each point the line makes a prediction $\hat y_i = w x_i + b$. The miss is the **residual**

$$e_i = y_i - \hat y_i \qquad\text{(the vertical distance from the point to the line)}$$

We cannot just add the residuals, because positive and negative misses cancel. So we **square** them and average:

$$\boxed{J(w,b) = \frac{1}{N}\sum_{i=1}^{N}\big(y_i - (w x_i + b)\big)^2}$$

$J$ is the *mean squared error* (MSE). Least squares means picking the $(w,b)$ that gives the **smallest** $J$.

Why square rather than use the absolute value?
1. Squaring removes the sign, so a miss above the line and a miss below it both count.
2. Big misses are penalised much more than small ones (2× the error gives 4× the cost).
3. $J$ becomes a smooth **quadratic bowl** in $(w,b)$. We can differentiate it, it has exactly one minimum, and that minimum has a closed-form solution. This is the main reason.
4. If the noise is Gaussian, the least-squares line is also the *maximum-likelihood* line.

> ⚠️ Least squares minimises **vertical** distances (errors in $y$ only), not the perpendicular distance to the line.
> Minimising perpendicular distance is a different method, called *total least squares* (or PCA).

The plan for this notebook:
1. Make some data
2. Look at the residuals and see the "squares" literally
3. Find the best line with a **plain for loop** (brute force)
4. Look at the cost landscape and see why it is a convex bowl
5. Use calculus to solve it exactly: normal equations, closed form, and one Newton step
6. Solve it iteratively with gradient descent, plus an animation
7. (Bridge to adaptive filters) The sample-by-sample version, **LMS**
8. What to do when the data is **not** linear

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ---- 1. Create data: a true line + Gaussian noise -------------------------
rng = np.random.default_rng(42)
N = 20
w_true, b_true = 0.8, 2.0               # hidden "truth" (in real life we don't know this)

x = np.sort(rng.uniform(0, 10, N))
y = w_true * x + b_true + rng.normal(0, 0.8, N)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, color="C0", zorder=3, label="measured data")
plt.plot([0, 10], [b_true, w_true*10 + b_true], "k--", lw=1, label="true line (hidden)")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Our noisy data"); plt.legend()
plt.show()

## 2. What does a "bad" line look like? Literally drawing the squares

Below we pick a line by guessing and draw, for each point:
* the residual $e_i$ (grey vertical segment)
* a square with side $|e_i|$. **Its area is $e_i^2$.**

The cost $J$ is the average area of the squares. Least squares means **making the total area of the squares as small as possible**.

In [ ]:
def predict(x, w, b):
    return w * x + b

def cost(w, b, x, y):
    e = y - predict(x, w, b)          # residuals
    return np.mean(e**2)              # mean of squared residuals

def draw_fit(ax, w, b, x, y, color="C3", squares=True, title=None):
    ax.scatter(x, y, color="C0", zorder=3, s=25)
    xx = np.array([-1, 11])
    ax.plot(xx, predict(xx, w, b), color=color, lw=2)
    e = y - predict(x, w, b)
    for xi, yi, ei in zip(x, y, e):
        y_hat = yi - ei
        ax.plot([xi, xi], [y_hat, yi], color="gray", lw=1)
        if squares:
            ax.add_patch(Rectangle((xi, min(yi, y_hat)), abs(ei), abs(ei),
                                   color=color, alpha=0.18, lw=0))
    ax.set_xlim(-1, 12); ax.set_ylim(-2, 13); ax.set_aspect("equal")
    ax.set_title(title or f"w={w:.2f}, b={b:.2f}  ->  J={cost(w, b, x, y):.2f}")

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
draw_fit(axs[0], 0.2, 6.0, x, y)       # a poor guess
draw_fit(axs[1], 0.8, 2.0, x, y, color="C2")  # close to the truth
plt.tight_layout(); plt.show()

## 3. The most intuitive optimiser: a for loop that tries everything

We don't need calculus yet. We can simply:
1. choose a range of candidate slopes $w$ and intercepts $b$
2. for **every** pair, compute the cost $J(w,b)$ (itself another loop over the data points)
3. keep the pair with the lowest cost

This is called a *grid search* (brute force). It is slow, but there is no magic in it at all. Every line of the loop follows directly from the definition of $J$.

In [ ]:
w_candidates = np.linspace(-1, 3, 201)     # 201 slopes to try
b_candidates = np.linspace(-4, 8, 201)     # 201 intercepts to try

best_J, best_w, best_b = np.inf, None, None
evaluations = 0

for w in w_candidates:                     # loop over slopes
    for b in b_candidates:                 # loop over intercepts
        # --- compute J(w, b) the "hand" way ---
        total = 0.0
        for xi, yi in zip(x, y):           # loop over data points
            residual = yi - (w * xi + b)
            total += residual ** 2
        J = total / N
        evaluations += 1
        # --- keep the best one so far ---
        if J < best_J:
            best_J, best_w, best_b = J, w, b

print(f"Tried {evaluations:,} lines")
print(f"Best line found: w = {best_w:.3f}, b = {best_b:.3f},  J = {best_J:.4f}")

It works, but look at the costs:
* The answer is only as precise as the grid spacing (here 0.02 in $w$ and 0.06 in $b$).
* $201 \times 201 \times 20 \approx 800\,000$ operations for **two** parameters. With $p$ parameters the grid grows like $201^p$, which is hopeless for anything larger.

We need to understand the **shape** of $J$ to do better.

## 4. The cost landscape: a bowl

Let's compute $J$ for every $(w,b)$ on the grid and plot it as a surface and as a contour map.

In [ ]:
W, B = np.meshgrid(w_candidates, b_candidates)
# vectorised cost for the whole grid: shape (len(b), len(w))
Jgrid = np.mean((y[None, None, :] - (W[..., None] * x[None, None, :] + B[..., None]))**2, axis=-1)

fig = plt.figure(figsize=(12, 4.8))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(W, B, Jgrid, cmap="viridis", alpha=0.85, linewidth=0)
ax1.scatter(best_w, best_b, best_J, color="red", s=40)
ax1.set_xlabel("w"); ax1.set_ylabel("b"); ax1.set_zlabel("J(w,b)")
ax1.set_title("Cost surface: one bowl, one bottom")

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contour(W, B, Jgrid, levels=np.geomspace(Jgrid.min()*1.05, Jgrid.max(), 25), cmap="viridis")
ax2.plot(best_w, best_b, "r*", ms=14, label="grid-search minimum")
ax2.set_xlabel("w"); ax2.set_ylabel("b"); ax2.set_title("Same bowl seen from above (contours)")
ax2.legend(); plt.tight_layout(); plt.show()

Two things to notice:
* There is **one** valley and **no** local minima to get stuck in. That is what *convex* means.
* The contours are **long, thin ellipses**. The bowl is steep in the $w$ direction and very flat in the $b$ direction. This becomes important for gradient descent in section 6.

## 5. Why it is convex, and how to solve it exactly

### First derivatives (gradient)
With $e_i = y_i - w x_i - b$:

$$\frac{\partial J}{\partial w} = -\frac{2}{N}\sum e_i\,x_i, \qquad
  \frac{\partial J}{\partial b} = -\frac{2}{N}\sum e_i$$

### Second derivatives (Hessian)
$$H = \begin{bmatrix} \frac{\partial^2 J}{\partial w^2} & \frac{\partial^2 J}{\partial w\,\partial b}\\[2pt]
                      \frac{\partial^2 J}{\partial b\,\partial w} & \frac{\partial^2 J}{\partial b^2}\end{bmatrix}
   = 2\begin{bmatrix} \overline{x^2} & \bar x\\ \bar x & 1\end{bmatrix}$$

(a bar means the average over the data).

* $H$ **does not depend on $w$ or $b$**. The curvature is the same everywhere, which is exactly what a quadratic bowl looks like.
* $\frac{\partial^2 J}{\partial w^2} = 2\overline{x^2} > 0$ and $\det H = 4(\overline{x^2}-\bar x^2) = 4\,\mathrm{Var}(x) > 0$.
  So $H$ is **positive definite** (both eigenvalues are positive), as long as not all $x_i$ are the same.
  This is the 2-D version of the "second derivative > 0" test. **$J$ is strictly convex and has exactly one minimum.**

### Set the gradient to zero (the *normal equations*)
$$\sum e_i = 0 \;\Rightarrow\; b = \bar y - w\bar x \qquad\qquad
  \sum e_i x_i = 0 \;\Rightarrow\; \boxed{w = \frac{\mathrm{Cov}(x,y)}{\mathrm{Var}(x)}},\quad \boxed{b = \bar y - w\,\bar x}$$

Read in plain words: the slope is how much $x$ and $y$ vary *together* divided by how much $x$ varies on its own, and the best line always passes through the **centre of mass** $(\bar x, \bar y)$ of the data.

### Matrix form (works for any number of parameters)
Stack a column of ones next to $x$: $X = [\,\mathbf{x}\;\; \mathbf 1\,]$, $\theta = [w,\,b]^T$. Then $J = \frac1N\|y - X\theta\|^2$ and

$$\nabla J = 0 \;\Rightarrow\; X^TX\,\theta = X^Ty \;\Rightarrow\; \theta^* = (X^TX)^{-1}X^Ty$$

(In code you use `np.linalg.lstsq` and never form the inverse explicitly; it is more accurate numerically.)

### "Use the second derivative": Newton's method
Newton's update is $\theta_{new} = \theta - H^{-1}\nabla J$. Because $J$ is *exactly* quadratic, a single Newton step lands **exactly** on the minimum, **from any starting point**. The closed form above is the same thing as that one Newton step.

In [ ]:
# --- closed form (statistics view) ---
w_cf = np.cov(x, y, bias=True)[0, 1] / np.var(x)
b_cf = y.mean() - w_cf * x.mean()

# --- matrix view ---
X = np.column_stack([x, np.ones(N)])
theta_ls, *_ = np.linalg.lstsq(X, y, rcond=None)

# --- Hessian: check positive definiteness ---
H = 2 * np.array([[np.mean(x**2), np.mean(x)],
                  [np.mean(x),    1.0       ]])
eig = np.linalg.eigvalsh(H)

# --- one Newton step from a silly starting point ---
def grad(w, b):
    e = y - (w * x + b)
    return np.array([-2*np.mean(e*x), -2*np.mean(e)])

theta0 = np.array([-5.0, 40.0])
theta_newton = theta0 - np.linalg.solve(H, grad(*theta0))

print(f"Grid search    : w = {best_w:.4f}, b = {best_b:.4f}")
print(f"Closed form    : w = {w_cf:.4f}, b = {b_cf:.4f}")
print(f"lstsq (matrix) : w = {theta_ls[0]:.4f}, b = {theta_ls[1]:.4f}")
print(f"1 Newton step  : w = {theta_newton[0]:.4f}, b = {theta_newton[1]:.4f}   (started at {theta0})")
print(f"np.polyfit     : {np.polyfit(x, y, 1).round(4)}")
print(f"\nHessian eigenvalues: {eig.round(3)}  -> both > 0, so the cost is convex")
print(f"Condition number  : {eig.max()/eig.min():.0f}  (ratio steep/flat; explains the thin ellipses)")
print(f"Minimum cost J*   : {cost(w_cf, b_cf, x, y):.4f}")

## 6. Doing it iteratively: gradient descent (with animation)

The closed form is perfect here, so why iterate? Because:
* for millions of parameters, solving $X^TX\theta=X^Ty$ becomes expensive,
* when data **streams in** (adaptive filters!) you cannot wait to collect all of it,
* most non-linear models have **no** closed form.

Gradient descent rolls downhill: $\;\theta \leftarrow \theta - \eta\,\nabla J(\theta)$.

For a quadratic bowl it converges if $\eta < 2/\lambda_{max}(H)$. Its speed is limited by the **flattest** direction $\lambda_{min}$.
That is why the thin ellipses matter: the path rushes down the steep wall and then crawls along the flat valley floor.

In [ ]:
eta = 0.02                                    # step size; must be < 2/lambda_max (try 0.025 and watch it explode!)
print(f"stability limit 2/lambda_max = {2/eig.max():.4f}")

w, b = -0.5, 7.5                              # deliberately bad start
history = [(w, b, cost(w, b, x, y))]

for it in range(1500):
    e = y - (w * x + b)                       # residuals for the current line
    dJ_dw = -2 * np.mean(e * x)
    dJ_db = -2 * np.mean(e)
    w -= eta * dJ_dw                          # step downhill
    b -= eta * dJ_db
    history.append((w, b, cost(w, b, x, y)))

history = np.array(history)
print(f"after {len(history)-1} iterations: w = {w:.4f}, b = {b:.4f}, J = {history[-1,2]:.4f}")
print(f"closed form                 : w = {w_cf:.4f}, b = {b_cf:.4f}, J = {cost(w_cf, b_cf, x, y):.4f}")

In [ ]:
# ---- animation: line fitting the data  +  the path on the cost bowl ----
frames = np.unique(np.concatenate([np.arange(0, 25), np.geomspace(25, len(history)-1, 35).astype(int)]))

fig, (axL, axR, axC) = plt.subplots(1, 3, figsize=(14, 4.4), dpi=72,
                                    gridspec_kw={"width_ratios": [1.15, 1, 1]})

axR.contour(W, B, Jgrid, levels=np.geomspace(Jgrid.min()*1.05, Jgrid.max(), 25), cmap="viridis", alpha=0.7)
axR.plot(w_cf, b_cf, "r*", ms=14)
path_line, = axR.plot([], [], "o-", color="C3", ms=3, lw=1.2)
axR.set_xlabel("w"); axR.set_ylabel("b"); axR.set_title("Path on the cost bowl")

axC.set_xlim(0, len(history)); axC.set_ylim(cost(w_cf, b_cf, x, y)*0.8, history[:,2].max()*1.5)
axC.set_yscale("log"); axC.set_xlabel("iteration"); axC.set_ylabel("J (log scale)")
axC.axhline(cost(w_cf, b_cf, x, y), color="r", ls="--", lw=1, label="least-squares minimum")
cost_line, = axC.plot([], [], color="C3"); axC.legend(); axC.set_title("Cost going down")
fig.tight_layout()

def update(k):
    i = frames[k]
    wi, bi, Ji = history[i]
    axL.clear()
    draw_fit(axL, wi, bi, x, y, title=f"iteration {i}:  w={wi:.2f}, b={bi:.2f},  J={Ji:.2f}")
    path_line.set_data(history[:i+1, 0], history[:i+1, 1])
    cost_line.set_data(np.arange(i+1), history[:i+1, 2])
    return path_line, cost_line

anim = FuncAnimation(fig, update, frames=len(frames), interval=120)
anim.save("least_squares_gd.gif", writer=PillowWriter(fps=8))   # GIF to put in your slides
plt.close(fig)
HTML(anim.to_jshtml())                                           # interactive player in the notebook

What to look for in the animation:
* **First few iterations:** the line swings around quickly. The slope $w$ is being corrected along the steep direction of the bowl.
* **Then:** the path turns and **crawls** along the flat valley floor. The intercept $b$ converges slowly, which is the ill-conditioning we predicted from the Hessian eigenvalues.
* **The squares** shrink until their total area cannot get any smaller.

💡 Tip: **centring** the data ($x \to x-\bar x$) makes $\bar x = 0$, so $H$ becomes diagonal, the ellipses become axis-aligned, and gradient descent gets much faster. Try it!

## 7. Bridge to adaptive filtering: from least squares to **LMS**

So far every update used **all** $N$ points (a *batch* gradient). In an adaptive filter the samples arrive one at a time, $n = 0, 1, 2, \dots$
The trick (Widrow & Hoff, 1960) is to use the gradient of **only the current sample's** squared error, $e[n]^2$, as a noisy estimate of the true gradient:

$$e[n] = y[n] - \big(w\,x[n] + b\big),\qquad
  w \leftarrow w + \mu\, e[n]\, x[n],\qquad b \leftarrow b + \mu\, e[n]$$

(The factor 2 is absorbed into $\mu$.) That is the **Least-Mean-Squares (LMS)** algorithm: the same cost and the same bowl, but descended in small, noisy, one-sample-at-a-time steps.
In a FIR filter, $x[n]$ becomes the vector of the last $M$ input samples and $w$ becomes the vector of filter taps. The update has exactly the same form.

| | Least Squares (LS) | LMS |
|---|---|---|
| Data used | all $N$ points at once | one sample per update |
| Solution | exact, one shot | approaches it and keeps jittering around it |
| Cost per update | solve $X^TX\theta = X^Ty$ | ~2 multiplications per weight |
| Tracks changing systems? | no (re-solve everything) | yes, that is its purpose |

(If you want exact LS solutions *recursively*, sample by sample, that is **RLS**, Recursive Least Squares.)

In [ ]:
mu = 0.008
w_l, b_l = -0.5, 7.5
w_hist, b_hist = [w_l], [b_l]

for epoch in range(400):                       # pass over the data repeatedly, like a stream
    for n in rng.permutation(N):
        e_n = y[n] - (w_l * x[n] + b_l)        # error on ONE sample
        w_l += mu * e_n * x[n]                 # LMS update
        b_l += mu * e_n
        w_hist.append(w_l); b_hist.append(b_l)

fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
axs[0].plot(w_hist, lw=0.8); axs[0].axhline(w_cf, color="r", ls="--", label="LS solution")
axs[0].set_title("LMS: slope w per sample update"); axs[0].set_xlabel("update n"); axs[0].legend()
axs[1].plot(b_hist, lw=0.8, color="C1"); axs[1].axhline(b_cf, color="r", ls="--", label="LS solution")
axs[1].set_title("LMS: intercept b per sample update"); axs[1].set_xlabel("update n"); axs[1].legend()
plt.tight_layout(); plt.show()
print(f"LMS final: w = {w_l:.3f}, b = {b_l:.3f}   |   LS: w = {w_cf:.3f}, b = {b_cf:.3f}")

Notice that LMS **never settles exactly**. Each update uses the gradient of a single sample, which is a noisy estimate, so $w$ keeps jittering around the LS solution.
A smaller $\mu$ means less jitter but slower convergence. This speed-versus-accuracy trade-off (called *misadjustment*) is a central design choice for adaptive filters.
Also notice that $b$ is again the slow one: the same ill-conditioning as in batch gradient descent.

## 8. When the data is **not** linear

Least squares will **always** give you a line, even when a line makes no sense. The method does not warn you. You have to check.

**The diagnostic tool is the residual plot.** If the model is right, the residuals look like random noise around zero.
If they show a **pattern** (a U-shape, a wave, a funnel), the model is missing something.

In [ ]:
# data from a curved (quadratic) relation
x2 = np.sort(rng.uniform(0, 10, 30))
y2 = 0.15 * x2**2 - 0.6 * x2 + 3 + rng.normal(0, 0.5, x2.size)

def ls_poly(x, y, deg):
    X = np.vander(x, deg + 1)                   # columns: x^deg ... x^1 x^0
    theta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return theta

def r2(y, yhat):
    return 1 - np.sum((y - yhat)**2) / np.sum((y - y.mean())**2)

xx = np.linspace(0, 10, 200)
fig, axs = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for col, deg in enumerate([1, 2]):
    th = ls_poly(x2, y2, deg)
    yhat = np.polyval(th, x2)
    axs[0, col].scatter(x2, y2, s=20)
    axs[0, col].plot(xx, np.polyval(th, xx), "C3", lw=2)
    axs[0, col].set_title(f"degree {deg} fit  |  MSE = {np.mean((y2-yhat)**2):.2f},  R² = {r2(y2, yhat):.3f}")
    axs[1, col].scatter(x2, y2 - yhat, s=20, color="C1")
    axs[1, col].axhline(0, color="k", lw=1)
    axs[1, col].set_title("residuals: " + ("clear U-shape -> model is wrong" if deg == 1 else "random -> model is fine"))
    axs[1, col].set_xlabel("x")
plt.tight_layout(); plt.show()

### An important correction: "linear" means linear **in the parameters**, not in $x$

The quadratic fit on the right is **still ordinary least squares**. We only added a column $x^2$ to the matrix $X$:

$$\hat y = \theta_2 x^2 + \theta_1 x + \theta_0 \quad\Rightarrow\quad X = [\,x^2\;\; x\;\; 1\,]$$

The model is curved in $x$, but it is **linear in $\theta$**. So the cost is still a convex quadratic bowl, the normal equations still hold, and the closed form still works.
The same goes for $\sin(x)$, $\log(x)$, $e^{x}$ or any other fixed *basis functions* you put into the columns.

So the honest statement is not "least squares cannot be used for non-linear data". It is:
**plain straight-line fitting fails for non-linear data, but least squares itself is still the workhorse. You change the features, not the method.**

### Beware: more flexibility is not always better (overfitting)

In [ ]:
# small training set + a separate test set from the same curve
x_tr = np.sort(rng.uniform(0, 10, 12));  y_tr = 0.15*x_tr**2 - 0.6*x_tr + 3 + rng.normal(0, 0.5, 12)
x_te = np.sort(rng.uniform(0, 10, 200)); y_te = 0.15*x_te**2 - 0.6*x_te + 3 + rng.normal(0, 0.5, 200)

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, deg in zip(axs, [1, 2, 9]):
    th = ls_poly(x_tr, y_tr, deg)
    mse_tr = np.mean((y_tr - np.polyval(th, x_tr))**2)
    mse_te = np.mean((y_te - np.polyval(th, x_te))**2)
    ax.scatter(x_tr, y_tr, s=25, zorder=3, label="training points")
    ax.plot(xx, np.polyval(th, xx), "C3", lw=2)
    ax.set_ylim(-2, 15)
    ax.set_title(f"degree {deg}\ntrain MSE = {mse_tr:.2f}   test MSE = {mse_te:.2f}")
axs[0].legend(); plt.tight_layout(); plt.show()

Degree 9 almost matches the training points but goes wild between them. Its **test** error is much worse. *Training error always goes down as you add parameters; the error on new data is what counts.*

### When the model is non-linear in its parameters
Example: a capacitor discharging, $v(t) = A\,e^{-t/\tau} + c$. The parameter $\tau$ sits *inside* the exponential, so no choice of columns for $X$ makes this linear.
You still minimise the sum of squared residuals (**non-linear least squares**), but:
* there is no closed form, so you iterate (Gauss–Newton / Levenberg–Marquardt, which `scipy.optimize.curve_fit` uses),
* the cost is **no longer guaranteed convex**, so a bad initial guess can get stuck in a local minimum.

In [ ]:
from scipy.optimize import curve_fit

t = np.linspace(0, 5, 40)
v = 4.0 * np.exp(-t / 1.2) + 0.5 + rng.normal(0, 0.08, t.size)

def model(t, A, tau, c):
    return A * np.exp(-t / tau) + c

params, _ = curve_fit(model, t, v, p0=[3, 1, 0])     # p0 = initial guess (needed!)
A_hat, tau_hat, c_hat = params

plt.figure(figsize=(6, 3.8))
plt.scatter(t, v, s=18, label="measured v(t)")
plt.plot(t, model(t, *params), "C3", lw=2, label=f"fit: A={A_hat:.2f}, tau={tau_hat:.2f}, c={c_hat:.2f}")
plt.plot(t, np.polyval(ls_poly(t, v, 1), t), "k--", lw=1, label="straight line (clearly wrong)")
plt.xlabel("t [s]"); plt.ylabel("v [V]"); plt.title("Non-linear least squares: RC discharge"); plt.legend()
plt.show()

## 9. Recommendations: what to do when a straight line doesn't fit

1. **Always plot the data and the residuals first.** A pattern in the residuals tells you the model is wrong. A single $R^2$ number can hide this.
2. **Transform the features** (still ordinary, convex LS):
   * polynomial terms $x^2, x^3$ for smooth curvature
   * $\log x$, $\sqrt{x}$, $\sin(\omega x)$, … when physics suggests them
3. **Transform the output** when the relation is multiplicative: $y = a e^{kx} \Rightarrow \ln y = \ln a + kx$ is a straight line in $(x, \ln y)$.
   (Careful: this changes how the noise is weighted.)
4. **Piecewise models / splines** when the behaviour changes in different regions.
5. **Non-linear least squares** (`curve_fit`, Levenberg–Marquardt) when you know the physical model and its parameters appear non-linearly. Give it a good initial guess.
6. **Control overfitting:** keep the model as simple as the residuals allow, check the error on held-out data (cross-validation), or add a penalty $\lambda\|\theta\|^2$ (*ridge regression*, still a closed form: $(X^TX+\lambda I)^{-1}X^Ty$).
7. **Outliers:** squaring makes one bad point pull the whole line. Use robust losses (Huber, absolute error) or RANSAC.
8. **Errors in $x$ as well as $y$?** Use total least squares (orthogonal regression), not ordinary LS.

---
### One-paragraph summary
Least squares picks the model parameters that make the **sum of squared vertical residuals** as small as possible. For any model that is linear in its parameters, this cost is a **convex quadratic bowl**. Its Hessian $2X^TX/N$ is constant and positive definite, so there is exactly one minimum. Setting the gradient to zero gives the **normal equations** $X^TX\theta = X^Ty$, which you can solve in one shot (closed form, the same as one Newton step) or approach iteratively (gradient descent). Doing that iteration one sample at a time is the **LMS** adaptive filter. If the data is curved, change the **features** and keep the method. Only when the parameters themselves enter non-linearly do you lose convexity and the closed form.